# Повтор проекта: новая матрица и новая анкета

## 1. Цель
Проверить сохранение прогноза МНК и отдельно обоснованность выдачи построчных анкет. Повторить расчёт с документированными аргументами для новой ключ-матрицы и новой анкеты без редактирования вычислительных ячеек. Обучающие и тестовые участники должны остаться теми же, что в первом проходе.

Ожидаю при обратимом P равенство R² до 1e-9 и максимальную разницу прогнозов ниже 1e-7 тыс. ₽. Это повтор знакомого проекта, а не новая независимая проверка обобщающего качества. Новый ключ не выбирается по тестовому качеству.

In [1]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sklearn
from run_project import run_project
RANDOM_STATE = 42
print("Python",sys.version.split()[0],"NumPy",np.__version__,"pandas",pd.__version__,"sklearn",sklearn.__version__)
print("Seed",RANDOM_STATE)


Python 3.12.14 NumPy 2.4.6 pandas 3.0.6 sklearn 1.8.0
Seed 42


**Вывод по цели.** На исходном проходе 5000 анкет дали одинаковый R² 0.371314, но известные пары раскрыли 100% профилей. Поэтому сохранение качества проверяю отдельно от приватности. Ни исходный, ни новый ключ не считаю достаточным основанием передавать реальные анкеты.

## 2. Данные и неизменное разбиение
Данные берутся по относительному пути ../../../technical/data-baseline либо COURSE_DATA; параметр data функции также принимает путь. В комплекте inputs/split-ids.csv сохранены исходные 3750 обучающих и 1250 тестовых member_id с порядком внутри частей. Функция сопоставляет их с текущим CSV по ключу, проверяет полноту, уникальность и отсутствие пересечения. Порядок строк входного CSV не определяет разбиение.

Собственный MyOLS использует lstsq на [1,X] без регуляризации. Для обратимого P можно явно перенести веса v=P⁻¹w и получить xPv+b=xw+b для любой строки. Для совпадения двух независимых fit на новых строках дополнительно проверяется полный ранг расширенной обучающей матрицы.

In [2]:
default_result=run_project(output=Path("artifacts/default"))
print(json.dumps(default_result,ensure_ascii=False,indent=2))


Python 3.12.14 numpy 2.4.6 pandas 3.0.6 sklearn 1.8.0
Случайность 42 данные /Users/dmitriikormin/Documents/Projects/ds-course/docs/reaudit-2026-11-16/ch09/fixes/student/retest/../../../technical/data-baseline
Строк/столбцов (5000, 6)
Пропуски {'member_id': 0, 'gender': 0, 'age': 0, 'income_k': 0, 'family_size': 0, 'spend_k': 0}
Полные дубли 0 повторные ID 0
Повторные профили 449
        gender      age  income_k  family_size  spend_k
count  5000.00  5000.00   5000.00      5000.00  5000.00
mean      0.55    35.65     97.87         2.62    19.38
std       0.50     9.69     38.86         1.06     7.73
min       0.00    18.00     25.00         1.00     0.00
25%       0.00    29.00     71.00         2.00    14.10
50%       1.00    36.00     91.00         3.00    19.30
75%       1.00    42.00    117.00         3.00    24.40
max       1.00    70.00    423.00         6.00    54.30
train/test 3750 1250 ранг P 4 cond 26.88453024045785
ранг [1,X_train] 5 максимальная ошибка восстановления 7.55105

**Вывод по данным и исходному ключу.** Функция проверила 5000 уникальных ID, разбиение 3750/1250 и полный ранг [1,X_train]. Исходный R² равен 0.371314, MAE 4.938463 тыс. ₽ против 6.184992 у среднего. Доход и годовые траты относятся к разным периодам; дата сбора учебных анкет не задана. Повторные профили разных людей не удаляются.

## 3. Новый ключ и новая строка через вход функции
Матрица и строка читаются из отдельных JSON. Передаю их параметрами существующей run_project; никаких коэффициентов или ячеек менять не нужно. Вторая анкета: мужчина 41 года, доход 110 тыс. ₽, семья из 4 человек. Проверяю восстановление на данных и совпадение прогнозов, а не объявляю некоторый диапазон cond универсально безопасным.

In [3]:
alternate_key=json.loads(Path("inputs/alternate-key.json").read_text())
alternate_member=json.loads(Path("inputs/alternate-member.json").read_text())
alternate_result=run_project(key=alternate_key,new_member=alternate_member,output=Path("artifacts/alternate"))
for filename in ["train-ids.csv","test-predictions.csv"]:
    old=pd.read_csv(Path("artifacts/default/internal")/filename)
    changed=pd.read_csv(Path("artifacts/alternate/internal")/filename)
    assert old.member_id.equals(changed.member_id)
assert abs(default_result["r2_open"]-alternate_result["r2_open"])<=1e-12
print("Участники и порядок обеих частей совпадают")
print(json.dumps(alternate_result,ensure_ascii=False,indent=2))


Python 3.12.14 numpy 2.4.6 pandas 3.0.6 sklearn 1.8.0
Случайность 42 данные /Users/dmitriikormin/Documents/Projects/ds-course/docs/reaudit-2026-11-16/ch09/fixes/student/retest/../../../technical/data-baseline
Строк/столбцов (5000, 6)
Пропуски {'member_id': 0, 'gender': 0, 'age': 0, 'income_k': 0, 'family_size': 0, 'spend_k': 0}
Полные дубли 0 повторные ID 0
Повторные профили 449
        gender      age  income_k  family_size  spend_k
count  5000.00  5000.00   5000.00      5000.00  5000.00
mean      0.55    35.65     97.87         2.62    19.38
std       0.50     9.69     38.86         1.06     7.73
min       0.00    18.00     25.00         1.00     0.00
25%       0.00    29.00     71.00         2.00    14.10
50%       1.00    36.00     91.00         3.00    19.30
75%       1.00    42.00    117.00         3.00    24.40
max       1.00    70.00    423.00         6.00    54.30
train/test 3750 1250 ранг P 4 cond 1.4395911605465732
ранг [1,X_train] 5 максимальная ошибка восстановления 5.6843

**Вывод по новому ключу.** При том же составе train/test новый ключ с cond=1.439591 сохранил R²0.371314; максимальная разница прогнозов1.92e-13тыс.₽. Новая анкета дала23.50тыс.₽, ошибочный ключ−407.61тыс.₽. Обратимая замена сохраняет линейный прогноз до заявленной вычислительной точности. Новая строка рассчитывается обеими обученными моделями, результаты сравниваются внутри функции. Число обусловленности и максимальные отклонения выведены выше. Отдельная диагностика плохого ключа и деревьев не является подбором параметров с новой независимой оценкой.

## 4. Открытые проверки переноса
Проверяю порядок ингредиентов по именам и новую строку вне обучающей линии зависимых признаков. Числа 23/16 для первого вопроса были раскрыты координатором до нахождения текста: самостоятельность числового ответа не заявляю. Объяснение JOIN/reindex и реализацию проверки делаю сам.

In [4]:
recipe=pd.Series([2.,1.],index=["кофе","молоко"])
price=pd.Series([3.,10.],index=["молоко","кофе"])
keyed=float(recipe.to_numpy() @ price.reindex(recipe.index).to_numpy())
reordered=float(recipe.to_numpy() @ price.iloc[::-1].reindex(recipe.index).to_numpy())
blind=float(recipe.to_numpy() @ price.to_numpy())
assert keyed==reordered==23. and blind==16.
row=np.array([0.,2.])
P=np.diag([1.,2.])
old_w=np.array([.5,.5])
transferred=np.linalg.solve(P,old_w)
refit_w=np.array([.2,.4])
answers=[float(row @ old_w),float((row @ P) @ transferred),float((row @ P) @ refit_w)]
assert np.allclose(answers,[1.,1.,1.6])
print("Порядок по ключам",keyed,reordered,"вслепую",blind)
print("Новая строка",row @ P,"старый/перенос/новый fit",answers)


Порядок по ключам 23.0 23.0 вслепую 16.0
Новая строка [0. 4.] старый/перенос/новый fit [1.0, 1.0, 1.6]


**Вывод по открытым вопросам.** JOIN по ingredient или reindex перед умножением сохраняет 23 ₽ при перестановке справочника, а голые позиции дают ошибочные 16 ₽. Для новой строки (0,2) после преобразования (0,4) прежний и явно перенесённый прогнозы равны 1, а новый fit из примера даёт 1.6: совпадение на зависимых обучающих строках недостаточно.

## 5. Отдельный температурный перенос
Если F=1.8C+32, то b+wC=(b−32w/1.8)+(w/1.8)F. Явный перенос коэффициента и intercept сохраняет прогноз физически той же температуры. Новое обучение обычного МНК без штрафа сохраняет множество достижимых прогнозов; для совпадения на произвольной новой строке нужна единственность расширенной системы. При одновременных C,F и intercept столбцы зависимы: на согласованных температурах прогнозы совпадают, а невозможная пара C=0,F=100 не получает такой гарантии. Ridge меняет штраф при смене шкалы. Этот перенос знаком по первому проходу; здесь он объяснён отдельно от кейса анкет.

## 6. Два вывода для получателя
**Сохранение модели.** Обычный МНК без штрафа при обратимой матрице может выразить ровно прежние линейные прогнозы. Внутри функции проверяются ранг, численное восстановление, связь весов, равенство тестового качества и новая анкета. При зависимых столбцах явный перенос надёжнее обещания о двух произвольных новых fit.

**Основания для выдачи анкет.** Смеси и сохранённое качество не доказывают конфиденциальность. По четырём независимым известным парам восстановлено 100% строк; копии одной пары не дают новых уравнений. Цель spend_k раскрыта в train. Если получателю выдать алгоритм и seed42, он воспроизведёт P без атаки по парам. Отсутствие полного ключа также не доказывает скрытость отдельного поля. Поэтому учебный contractor-demo нельзя выдавать за обоснованное обезличивание реальных данных.

Внутренний пакет: ID/метки/прогнозы, key.npy, модель, открытая новая анкета, summary, receiver-information и письмо. Пакет contractor-demo: только смешанные train/test/new и train spend_k, без member_id и ключа. Исходный код и README — внутренний комплект воспроизведения; если передать его со стандартным seed подрядчику, ключ перестаёт быть неизвестным. Никакой внешней отправки не выполнялось.